# Machine Learning — Lab 11
## K-Means Clustering and Cluster Interpretation

**Main Course Learning Outcomes — CLO2, CLO4, CLO5**

- **CLO2:** Analyze datasets and apply appropriate preprocessing, transformation, and feature engineering techniques.
- **CLO4:** Apply supervised and unsupervised learning techniques to solve practical problems and interpret their outcomes.
- **CLO5:** Evaluate machine learning models using appropriate metrics and justify decisions based on performance trade-offs and real-world context.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

> **Assessment principle:** Producing colored clusters is not enough. Full credit requires you to explain **why distances are meaningful, how centroids move, why scaling matters, how $K$ is justified, and what the clusters actually mean in the application context**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Manual K-Means iteration | 30 min | Compute distances, assignments, centroids, and WCSS |
| 2. Scaled customer clustering | 20 min | Standardize features and fit K=4 |
| 3. Choose K | 20 min | Use inertia, silhouette, and interpretability |
| 4. Initialization & scaling | 20 min | Study random starts and raw-vs-scaled clustering |
| 5. Visualization & interpretation | 15 min | Profile clusters and inspect a 2D view |
| 6. Debugging, reflection & viva | 15 min | Correct cluster-label misconceptions and explain limitations |
| **Total** | **120 min** | |

## Learning Objectives

By the end of this lab, you should be able to:

1. explain K-Means as an unsupervised clustering algorithm;
2. compute Euclidean distance manually;
3. perform one assignment step;
4. recompute centroids manually;
5. explain the K-Means objective;
6. standardize features before distance-based clustering;
7. compare different values of $K$;
8. interpret inertia and silhouette score;
9. explain why initialization can change the result;
10. profile and interpret clusters using original feature units.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

print("Machine Learning Lab 11 environment ready.")

# Part I — K-Means from First Principles

K-Means divides data into $K$ clusters.

Each cluster is represented by a centroid:

$$
\mu_1,\mu_2,\ldots,\mu_K.
$$

The algorithm alternates between:

### Assignment step

$$
c_i
=
\arg\min_k
\|x_i-\mu_k\|^2
$$

### Update step

$$
\mu_k
=
\frac{1}{|C_k|}
\sum_{x_i\in C_k}
x_i.
$$

In [ ]:
tiny = pd.DataFrame({
    "x1": [1.0, 1.0, 2.0, 8.0, 9.0, 8.0],
    "x2": [1.0, 2.0, 1.0, 8.0, 8.0, 9.0],
}, index=["A", "B", "C", "D", "E", "F"])

display(tiny)

centroids = np.array([
    [1.0, 1.0],
    [8.0, 8.0],
])

print("Initial centroids:")
print(centroids)

## Task 1.1 — Implement Euclidean Distance

In [ ]:
def euclidean_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    # TODO: implement Euclidean distance.
    distance = None

    return float(distance)

assert abs(
    euclidean_distance(
        np.array([0.0, 0.0]),
        np.array([3.0, 4.0])
    ) - 5.0
) < 1e-12

print("Euclidean-distance test passed.")

## Task 1.2 — Manual Assignment Step

Before running the next cell:

1. calculate the distance from point B $(1,2)$ to both centroids;
2. calculate the distance from point E $(9,8)$ to both centroids;
3. predict the cluster assignment for all six points.

**Your prediction / justification:**

In [ ]:
assignment_rows = []

for label, row in tiny.iterrows():
    point = row.to_numpy()

    d1 = euclidean_distance(point, centroids[0])
    d2 = euclidean_distance(point, centroids[1])

    cluster = int(np.argmin([d1, d2]))

    assignment_rows.append({
        "point": label,
        "x1": point[0],
        "x2": point[1],
        "distance_to_c1": d1,
        "distance_to_c2": d2,
        "assigned_cluster": cluster,
    })

assignment_table = pd.DataFrame(assignment_rows)
display(assignment_table.round(4))

## Task 1.3 — Implement Centroid Update

Complete a function that receives points, cluster assignments, and $K$, then returns the updated centroids.

In [ ]:
def recompute_centroids(X, labels, K):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels, dtype=int)

    new_centroids = []

    for k in range(K):
        # TODO: select cluster points and compute their mean.
        cluster_points = None
        centroid = None
        new_centroids.append(centroid)

    return np.asarray(new_centroids, dtype=float)

tiny_labels = assignment_table["assigned_cluster"].to_numpy()

updated_centroids = recompute_centroids(
    tiny.to_numpy(),
    tiny_labels,
    K=2
)

expected = np.array([
    [4/3, 4/3],
    [25/3, 25/3],
])

assert np.allclose(updated_centroids, expected)

print("Centroid-update test passed.")
print(updated_centroids)

## Task 1.4 — Interpret the Update

Answer:

1. Why did each centroid move?
2. What does the new centroid represent?
3. If assignments do not change in the next iteration, what happens?
4. Why is K-Means iterative rather than one-step?

**Your answer / interpretation:**

# Part II — K-Means Objective

K-Means tries to minimize:

$$
WCSS
=
\sum_{k=1}^{K}
\sum_{x_i\in C_k}
\|x_i-\mu_k\|^2.
$$

This is often called **inertia**.

## Task 2.1 — Compute and Interpret WCSS

Complete `compute_wcss` and calculate the WCSS for the updated tiny-data centroids. Explain what a lower WCSS means and why WCSS usually decreases when the number of clusters increases.

**Your calculation / reasoning:**

In [ ]:
def compute_wcss(X, labels, centroids):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels, dtype=int)
    centroids = np.asarray(centroids, dtype=float)

    # TODO: sum squared distances from each point to its assigned centroid.
    total = None

    return float(total)

wcss_value = compute_wcss(
    tiny.to_numpy(),
    tiny_labels,
    updated_centroids
)

assert wcss_value >= 0
print("Tiny-data WCSS:", round(wcss_value, 4))

# Part III — Customer Segmentation

Use the personalized customer dataset. Inspect feature ranges, standardize the features, fit $K=4$, inverse-transform the centroids, and create short descriptive cluster profiles.

In [ ]:
rng = np.random.default_rng(3452)

segments = []
specs = [
    (4200, 8.5, 45, 12, 20),
    (10500, 6.0, 150, 9, 15),
    (2200, 1.5, 70, 75, 4),
    (7600, 2.8, 240, 28, 8),
]
sizes = [150, 130, 140, 120]

for spend, visits, basket, recency, digital in specs:
    n_i = sizes[len(segments)]
    block = pd.DataFrame({
        "annual_spend": rng.normal(spend, spend * 0.12, n_i),
        "visits_per_month": rng.normal(visits, 1.0, n_i),
        "avg_basket": rng.normal(basket, basket * 0.13, n_i),
        "days_since_last_purchase": rng.normal(recency, max(4, recency * 0.15), n_i),
        "digital_sessions": rng.normal(digital, 2.5, n_i),
    })
    segments.append(block)

customer_master = pd.concat(segments, ignore_index=True)

customer_master["annual_spend"] = customer_master["annual_spend"].clip(lower=300)
customer_master["visits_per_month"] = customer_master["visits_per_month"].clip(lower=0.1)
customer_master["avg_basket"] = customer_master["avg_basket"].clip(lower=5)
customer_master["days_since_last_purchase"] = customer_master["days_since_last_purchase"].clip(lower=0)
customer_master["digital_sessions"] = customer_master["digital_sessions"].clip(lower=0)

customer_master = customer_master.round({
    "annual_spend": 2,
    "visits_per_month": 2,
    "avg_basket": 2,
    "days_since_last_purchase": 1,
    "digital_sessions": 1,
})

print("Master customer dataset shape:", customer_master.shape)
display(customer_master.head())

## Task 3.1 — Personalized Working Dataset

Enter the last four digits of your student ID. Your value determines a reproducible sample of 440 customers.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 11000 + (STUDENT_ID_LAST4 % 1000)

customers = customer_master.sample(
    n=440,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

print("Your clustering seed:", SEED)
print("Working dataset shape:", customers.shape)

## Task 3.2 — Predict Before Clustering

Answer:

1. Which feature has the largest numerical scale?
2. Which feature has the smallest scale?
3. Why might clustering raw values be misleading?
4. Which features do you expect to distinguish active and inactive customers?

**Your prediction / justification:**

In [ ]:
summary = customers.describe().T[["mean", "std", "min", "max"]]
summary["range"] = summary["max"] - summary["min"]
display(summary.round(2))

# Part IV — Standardize Before K-Means

Because K-Means uses Euclidean distance, feature scaling matters.

$$
z=\frac{x-\mu}{\sigma}.
$$

## Task 4.1 — Standardize the Customer Features

Standardize the five customer features and verify that their transformed means are approximately 0 and standard deviations approximately 1.

Explain why this step is important before Euclidean-distance-based clustering.

**Your answer / interpretation:**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customers)

scaled_df = pd.DataFrame(
    X_scaled,
    columns=customers.columns
)

scale_check = pd.DataFrame({
    "mean": scaled_df.mean(),
    "std": scaled_df.std(ddof=0),
})

display(scale_check.round(4))

# Part V — Fit K-Means with $K=4$

## Task 5.1 — Fit K-Means with $K=4$

Fit K-Means using `k-means++` and multiple initializations. Report cluster sizes, inertia, and silhouette score.

Then inspect the inverse-transformed centroids and write a short feature-based profile for each cluster.

**Your answer / interpretation:**

In [ ]:
kmeans4 = KMeans(
    n_clusters=4,
    init="k-means++",
    n_init=20,
    random_state=SEED
)

labels4 = kmeans4.fit_predict(X_scaled)

print("Cluster counts:")
display(pd.Series(labels4).value_counts().sort_index().to_frame("count"))

print("Inertia:", round(kmeans4.inertia_, 3))
print("Silhouette score:", round(silhouette_score(X_scaled, labels4), 4))

# Part VI — Interpret Centroids in Original Units

In [ ]:
centroids_original = pd.DataFrame(
    scaler.inverse_transform(kmeans4.cluster_centers_),
    columns=customers.columns
)

centroids_original.index.name = "cluster"
display(centroids_original.round(2))

clustered_customers = customers.copy()
clustered_customers["cluster"] = labels4

profile = clustered_customers.groupby("cluster").mean(numeric_only=True)
display(profile.round(2))

## Task 6.1 — Choose the Number of Clusters

Compare $K=2$ through $6$ with inertia and silhouette. Choose one defensible $K$ and justify it using both numerical evidence and interpretability.

**Your answer / interpretation:**

In [ ]:
k_values = list(range(2, 7))
rows = []
models = {}

for k in k_values:
    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        random_state=SEED
    )

    labels = model.fit_predict(X_scaled)

    rows.append({
        "K": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
    })
    models[k] = model

k_table = pd.DataFrame(rows)
display(k_table.round(4))

plt.figure(figsize=(7, 4))
plt.plot(k_table["K"], k_table["inertia"], marker="o")
plt.xlabel("K")
plt.ylabel("Inertia / WCSS")
plt.title("Elbow Method")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(k_table["K"], k_table["silhouette"], marker="o")
plt.xlabel("K")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score by K")
plt.show()

## Task 6.2 — Initialization and Representation

Run only three random single-start initializations, then compare scaled vs unscaled K-Means. Explain local-optimum sensitivity and why raw large-scale features can dominate Euclidean distance.

**Your answer / interpretation:**

In [ ]:
stability_rows = []

for rs in [1, 19, 57]:
    model = KMeans(
        n_clusters=4,
        init="random",
        n_init=1,
        random_state=rs
    )

    labels = model.fit_predict(X_scaled)
    counts = np.bincount(labels, minlength=4)

    stability_rows.append({
        "random_state": rs,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "cluster_sizes": tuple(counts.tolist()),
    })

stability_table = pd.DataFrame(stability_rows)
display(stability_table)

kmeans_raw = KMeans(
    n_clusters=4,
    init="k-means++",
    n_init=20,
    random_state=SEED
)

raw_labels = kmeans_raw.fit_predict(customers)

raw_silhouette = silhouette_score(customers, raw_labels)
scaled_silhouette = silhouette_score(X_scaled, labels4)

comparison = pd.DataFrame({
    "Raw features": {"silhouette": raw_silhouette},
    "Standardized features": {"silhouette": scaled_silhouette},
}).T

display(comparison.round(4))

# Part VII — Two-Dimensional Visualization

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    customers["annual_spend"],
    customers["visits_per_month"],
    c=labels4,
    alpha=0.75
)
plt.xlabel("Annual Spend")
plt.ylabel("Visits per Month")
plt.title("K-Means Clusters — 2D View of a 5D Model")
plt.show()

## Task 7.1 — Interpret the Two-Dimensional Cluster View

Use the plot above to describe the visible cluster structure. Explain why overlap in this 2D view does not necessarily mean that the full five-dimensional clustering is poor.

**Your answer / interpretation:**

## Task 7.2 — Debugging Check

Cluster labels 0, 1, 2, ... are arbitrary. Explain why “Cluster 3 is higher than Cluster 2” is meaningless unless a specific centroid/profile comparison supports the statement.

**Your answer / interpretation:**

# Individual Understanding Check

Your instructor may select one question for a 60–90 second explanation.

1. What are the assignment and update steps in K-Means?
2. What does inertia measure?
3. Why must scaling be considered before K-Means?
4. Why are cluster numbers not ordered categories?

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. What result in this lab most changed or confirmed your expectation?
2. What modeling mistake would most seriously invalidate the result?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that you have completed all required TODOs, executed the required experiment(s), interpreted the main result, answered the reflection questions, and can explain your work individually.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Trace / prediction / debugging | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |